In [3]:
import pandas as pd
import holidays
import os

PATH = "../data/raw/"

#Chargement de tables Olist
orders = pd.read_csv(os.path.join(PATH, "olist_orders_dataset.csv"))
reviews = pd.read_csv(os.path.join(PATH, "olist_order_reviews_dataset.csv"))
customers = pd.read_csv(os.path.join(PATH, "olist_customers_dataset.csv"))
items = pd.read_csv(os.path.join(PATH, "olist_order_items_dataset.csv"))
products = pd.read_csv(os.path.join(PATH, "olist_products_dataset.csv"))
payments = pd.read_csv(os.path.join(PATH, "olist_order_payments_dataset.csv"))
sellers = pd.read_csv(os.path.join(PATH, "olist_sellers_dataset.csv"))
geo = pd.read_csv(os.path.join(PATH, "olist_geolocation_dataset.csv"))
translation = pd.read_csv(os.path.join(PATH, "product_category_name_translation.csv"))

#Chargement de la source externe 
economy = pd.read_excel(os.path.join(PATH, "Cities_Brazil_IBGE.xlsx"))

all_dfs = {
    "Orders": orders, "Reviews": reviews, "Customers": customers,
    "Items": items, "Products": products, "Payments": payments,
    "Sellers": sellers, "Geo": geo, "Translation": translation,
    "IBGE_Economy": economy
}

for name, df in all_dfs.items():
    print(f"{name:15} | Lignes: {df.shape[0]:<10} | Colonnes: {df.shape[1]}")

print("\n--- TOUTES LES SOURCES SONT PRÊTES POUR L'AUDIT ---")

Orders          | Lignes: 99441      | Colonnes: 8
Reviews         | Lignes: 99224      | Colonnes: 7
Customers       | Lignes: 99441      | Colonnes: 5
Items           | Lignes: 112650     | Colonnes: 7
Products        | Lignes: 32951      | Colonnes: 9
Payments        | Lignes: 103886     | Colonnes: 5
Sellers         | Lignes: 3095       | Colonnes: 4
Geo             | Lignes: 1000163    | Colonnes: 5
Translation     | Lignes: 71         | Colonnes: 2
IBGE_Economy    | Lignes: 5570       | Colonnes: 14

--- TOUTES LES SOURCES SONT PRÊTES POUR L'AUDIT ---


In [4]:
#Audit des Valeurs Manquantes 
print(" ANALYSE DES VALEURS MANQUANTES (en %)")
for name, df in all_dfs.items():
    null_pct = df.isnull().mean() * 100
    if null_pct.sum() > 0:
        print(f"\n--- {name} ---")
        print(null_pct[null_pct > 0].round(2))

# Audit des Doublons
print("\n\n ANALYSE DES DOUBLONS")
for name, df in all_dfs.items():
    dup_count = df.duplicated().sum()
    print(f"{name:15} | Doublons détectés : {dup_count}")

 ANALYSE DES VALEURS MANQUANTES (en %)

--- Orders ---
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
dtype: float64

--- Reviews ---
review_comment_title      88.34
review_comment_message    58.70
dtype: float64

--- Products ---
product_category_name         1.85
product_name_lenght           1.85
product_description_lenght    1.85
product_photos_qty            1.85
product_weight_g              0.01
product_length_cm             0.01
product_height_cm             0.01
product_width_cm              0.01
dtype: float64

--- IBGE_Economy ---
IDHM    0.11
dtype: float64


 ANALYSE DES DOUBLONS
Orders          | Doublons détectés : 0
Reviews         | Doublons détectés : 0
Customers       | Doublons détectés : 0
Items           | Doublons détectés : 0
Products        | Doublons détectés : 0
Payments        | Doublons détectés : 0
Sellers         | Doublons détectés : 0
Geo             | Doublons détectés : 261831
Translat

In [6]:
# Traitement des doublons Geo 
geo_clean = geo.drop_duplicates(subset=['geolocation_zip_code_prefix']).copy()

# Imputation des catégories de produits manquantes
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Suppression des commandes sans date de livraison (Inutilisables pour le retard)
orders_clean = orders.dropna(subset=['order_delivered_customer_date']).copy()

print(f"Audit Clean : Geo ({len(geo_clean)} lignes), Orders ({len(orders_clean)} lignes)")

Audit Clean : Geo (19015 lignes), Orders (96476 lignes)


In [5]:
# Conversion des colonnes en format Datetime 
date_columns = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# Vérification de la cohérence logique
# Une livraison ne peut pas arriver avant l'achat.
ghost_data = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]
print(f"Nombre de commandes 'fantômes' (livrées avant achat) : {len(ghost_data)}")

# 3. Vérification des Trous de liaison
# Est-ce que toutes les commandes ont des articles 
orders_without_items = orders[~orders['order_id'].isin(items['order_id'])]
print(f"Commandes sans aucun article associe : {len(orders_without_items)}")

Nombre de commandes 'fantômes' (livrées avant achat) : 0
Commandes sans aucun article associe : 775


In [7]:
# Résolution de la granularité : On agrège la table Items par commande
# Pour n'avoir qu'une seule ligne par commande dans notre base finale.
items_agg = items.groupby('order_id').agg({
    'price': 'sum',               
    'freight_value': 'sum',       
    'product_id': 'count'        
}).rename(columns={'product_id': 'qty_items'}).reset_index()

# Fusion des tables Olist (Coeur du projet)
# On utilise un 'inner' join pour Items afin d'éliminer automatiquement les 775 commandes sans articles
master = orders_clean.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
master = master.merge(customers[['customer_id', 'customer_city', 'customer_state']], on='customer_id', how='left')
master = master.merge(items_agg, on='order_id', how='inner') 

# Préparation et Fusion avec la Source 3 (IBGE Economy)
master['customer_city'] = master['customer_city'].str.lower().str.strip()
economy['LocalCidade'] = economy['LocalCidade'].str.lower().str.strip()

master = master.merge(
    economy[['LocalCidade', 'LocalUF', 'Pib_2014', 'PopEstimada_2018']], 
    left_on=['customer_city', 'customer_state'], 
    right_on=['LocalCidade', 'LocalUF'], 
    how='left'
)

# Ajout de la Source 2 : Jours Fériés (Holidays)
br_holidays = holidays.Brazil()
master['is_holiday'] = master['order_purchase_timestamp'].dt.date.apply(lambda x: x in br_holidays).astype(int)

# Création de la Cible 
master['is_late'] = (master['order_delivered_customer_date'] > master['order_estimated_delivery_date']).astype(int)

# Suppression des colonnes de jointure inutiles
master = master.drop(columns=['LocalCidade', 'LocalUF'])

# Sauvegarde du travail 
master.to_csv("../data/processed/master_data.csv", index=False)

print(f"Phase 2 terminee avec succes")
print(f"Dimensions du Master Dataset : {master.shape[0]} lignes et {master.shape[1]} colonnes.")
print(f"Taux de retard réel (is_late) : {master['is_late'].mean():.2%}")
print(f"Couverture PIB (données non nulles) : {master['Pib_2014'].notna().mean():.2%}")

Phase 2 terminee avec succes
Dimensions du Master Dataset : 97005 lignes et 18 colonnes.
Taux de retard réel (is_late) : 8.11%
Couverture PIB (données non nulles) : 99.12%


### Évaluation des Biais et Cohérence
*   **Données manquantes structurelles :** Environ 3% des commandes n'ont pas de date de livraison réelle.Nous avons choisi de les exclure pour garantir la fiabilité du calcul des retards.
*   **Données Fantômes :** L'audit de cohérence temporelle affiche 0 erreur (aucune livraison avant achat). La chronologie du dataset est saine.
*   **Biais Géographique :** On observe une hyper-concentration sur l'État de São Paulo (SP). Le modèle devra prendre en compte cette disparité spatiale pour ne pas biaiser les prédictions vers les zones urbaines uniquement.

### Dictionnaire de Données (Extraits principaux)

| Nom de la colonne | Type | Description | Source | Valeurs |
| :--- | :--- | :--- | :--- | :--- |
| `order_id` | Object | Identifiant unique de commande | Olist_Orders | UUID |
| `is_late` | Int (0/1) | Cible (Target) : 1 si livraison réelle > estimée | Calculé | 0, 1 |
| `review_score` | Int | Note de satisfaction client | Olist_Reviews | 1 à 5 |
| `product_category_name` | String | Catégorie du produit | Olist_Products | 74 types |
| `is_holiday` | Int (0/1) | 1 si la commande est passée un jour férié | Holidays Lib | 0, 1 |
| `Pib_2014` | Float | PIB de la ville du client | IBGE_Economy | Numérique |